<figure>
    <picture>
        <source srcset="assets/jii-logo.png" style="max-height: 10rem;">
        <img src="https://raw.githubusercontent.com/Computational-Biology-Aachen/2026-photosynthesis-hackathon-template/refs/heads/main/assets/jii-logo.png" style="max-height: 10rem;">
    </picture>
</figure>

# Ambit data

This notebook contains example analysis for the ambit data kindly supplied by the JII

| Column        | Description                                                                            |
| ------------- | -------------------------------------------------------------------------------------- |
| Type          | Sensor type, one of `Par`, `Leaf`, `Wasp`, `SPACER`, `SS`, `MPF2`, `FI`, `qE1`, `Rec1` |
| PAR           | Photosynthetically active radiation                                                    |
| Time          | Full datetime information                                                              |
| calendar_date | Just date                                                                              |
| Day           | Day of week as 0-6                                                                     |
| Act_time      | Time of actinic light <span style="color: red">?</span>                                |
| PTS           | <span style="color: red">?</span>                                                      |
| Sun           | Background light by sun                                                                |
| Leaf          | <span style="color: red">?</span>                                                      |
| Actinic       | Background light between measurement pulses                                            |
| Temp          | Temperature                                                                            |
| Full          | bool                                                                                   |
| ToD           | Time of day <span style="color: red">?</span>                                          |
| Fmp           | Highest light-adapted fluorescence; aka Fm' (Fm prime)                                     |
| Fs            | Lowest light-adapted fluorescence                                                      |
| F0            | Lowest fluorescence                                                                    |
| FM            | Highest fluorescence                                                                   |
| SigF          | Raw Fluorescence Signal                                                                |
| Fluo          | Normalised fluorescence signal                                                         |
| RefF          | <span style="color: red">?</span>                                                     |
| Count         | <span style="color: red">?</span>                                                      |


In [ ]:
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.data import load_jii_ambit


def split_df_by_sensor(df: pd.DataFrame) -> Iterable[pd.DataFrame]:
    gb = df.groupby(df["Type"])
    return (
        drop_all_na_columns(gb.get_group(i).drop(columns=["Type"]))
        for i in ["FI", "Leaf", "MPF2", "Par", "Rec1", "SPACER", "SS", "Wasp", "qE1"]
    )


def drop_all_na_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.isna().all(axis=0)]


df = load_jii_ambit(Path("data"))

# The df is a collection of multiple different sensors, so a lot of data is NaN.
# if you want to analyse each sensor individually without all the noise, you can use
# these dataframes
(
    fi,
    leaf,
    mpf2,
    par,
    rec1,
    spacer,
    ss,
    wasp,
    qe1,
) = split_df_by_sensor(df)

## EDA

[matplotlib](https://matplotlib.org/stable/) is excessively slow for plotting large amounts of data, so we suggest using [plotly](https://plotly.com/python/) instead.

In [ ]:
# Filter out null values for raw fluorescence signals
df_plot = df[df["SigF"].notna() & df["Time"].notna()].copy()

print(f"Plotting {len(df_plot):,} data points with valid SigF and Time values")

# Create interactive Plotly figure
fig = (
    go.Figure(
        layout=go.Layout(
            width=800,
            height=500,
            title="Raw Fluorescence Signal (SigF) vs Time<br /><sub>2+ days of continuous measurements</sub>",
            xaxis_title="Time",
            yaxis_title="SigF - Fluorescence Signal (AU)",
            hovermode="x unified",
            showlegend=True,
        )
    )
    .add_trace(
        go.Scatter(
            x=df_plot["Time"],
            y=df_plot["SigF"],
            mode="lines",
            name="SigF (Signal)",
            line={"color": "blue", "width": 1},
            hovertemplate="Time: %{x}<br>SigF: %{y:.1f} AU<extra></extra>",
        )
    )
    .update_xaxes(tickformat="%Y-%m-%d %H:%M", tickangle=-45)
)

fig.show()

We can further split the raw fluorescence signal by sensor

In [ ]:
fig = go.Figure(
    layout=go.Layout(
        width=800,
        height=500,
        title="Raw Fluorescence Signal by sensor",
        xaxis_title="Time",
        yaxis_title="SigF - Fluorescence Signal (AU)",
        hovermode="x unified",
        showlegend=True,
    )
).update_xaxes(tickformat="%Y-%m-%d %H:%M", tickangle=-45)

for sensor, name in (
    (fi, "fi"),
    (mpf2, "mpf2"),
    (rec1, "rec1"),
    (spacer, "spacer"),
    (ss, "ss"),
    (wasp, "wasp"),
    (qe1, "qe1"),
):
    fig.add_trace(
        go.Scatter(
            x=sensor["Time"],
            y=sensor["SigF"],
            mode="lines",
            name=name,
        )
    )

fig.show()